In [ ]:
import wfdb
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from scipy.signal import butter, filtfilt
import ruptures as rpt

selected_records = ['slp01a', 'slp01b', 'slp02a', 'slp02b']
fs_default = 250  # sampling rate, just in case

def bandpass_filter(signal, lowcut=0.5, highcut=40, fs=fs_default, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal)

def smooth_signal(signal, window=5):
    return pd.Series(signal).rolling(window, center=True, min_periods=1).mean().to_numpy()

def get_rr_intervals(ecg, fs):
    """Enkel R-topp deteksjon med peak-finding"""
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(ecg, distance=0.6*fs, height=np.mean(ecg))
    rr_intervals = np.diff(peaks) / fs  # i sekunder
    return rr_intervals, peaks

def hrv_metrics(rr_intervals):
    sdnn = np.std(rr_intervals)
    rmssd = np.sqrt(np.mean(np.diff(rr_intervals)**2))
    nn50 = np.sum(np.abs(np.diff(rr_intervals)) > 0.05)
    pnn50 = nn50 / len(rr_intervals) * 100
    return sdnn, rmssd, nn50, pnn50

def rsa_corr(rr_intervals, resp_signal, peaks):
    # Match resp signal til R-topp tidspunkter
    rr_times = peaks[:-1]
    resp_at_rr = resp_signal[rr_times]
    return np.corrcoef(rr_intervals, resp_at_rr)[0,1]

for rec in selected_records:
    print(f"\nProcessing {rec}")
    record = wfdb.rdrecord(f'./mit-bih-polysomnographic-database-1.0.0/mit-bih-polysomnographic-database-1.0.0/{rec}')  # juster path
    fs = record.fs
    ecg = record.p_signal[:,0]
    # Hvis respirationskanal finnes
    if record.p_signal.shape[1] > 1:
        resp = record.p_signal[:,1]
    else:
        resp = np.sin(np.linspace(0, 10*np.pi, len(ecg)))  # dummy resp hvis ikke

    # Filtrering og glatting
    ecg_filt = bandpass_filter(ecg, fs=fs)
    resp_smooth = smooth_signal(resp)

    # RR-intervaller
    rr_intervals, peaks = get_rr_intervals(ecg_filt, fs)
    sdnn, rmssd, nn50, pnn50 = hrv_metrics(rr_intervals)
    rsa = rsa_corr(rr_intervals, resp_smooth, peaks)

    print(f"HRV (SDNN): {sdnn:.3f}, RMSSD: {rmssd:.3f}, NN50: {nn50}, pNN50: {pnn50:.1f}%")
    print(f"RSA (RR-Resp korrelasjon): {rsa:.3f}")
    print(f"Resp (mean ± std): {np.mean(resp_smooth):.3f} ± {np.std(resp_smooth):.3f}")

    # =========================
    # Change Point Detection
    # =========================
    # Vi bruker RR-intervaller som signal for CPD
    # =========================
# CPD på RR-intervaller
# =========================
model = "l2"
algo = rpt.Pelt(model=model).fit(rr_intervals)
result = algo.predict(pen=0.03)  # CPD-punkter i rr_intervals-array

# Konverter til sample-indekser i EKG
# peaks[:-1] er starttidspunktene til RR-intervallene i samples
cp_samples = [peaks[cp-1] for cp in result[:-1]]  # ekskluder siste som alltid er end

# =========================
# Plot med stjerner på riktig tid
# =========================
fig = go.Figure()
fig.add_trace(go.Scatter(y=ecg_filt, name='EKG filtrert', line=dict(color='red')))
fig.add_trace(go.Scatter(y=resp_smooth, name='Resp glattet', line=dict(color='blue'), yaxis="y2"))

# Markér CPD under signalet
for cp in cp_samples:
    fig.add_trace(go.Scatter(x=[cp], y=[np.min(ecg_filt)-0.1],  # litt under min EKG
                             mode='markers', marker=dict(color='green', size=10, symbol='star'),
                             name='CPD'))

fig.update_layout(
    title=f'{rec} - EKG og Resp + CPD',
    xaxis_title='Samples',
    yaxis=dict(title='EKG (mV)'),
    yaxis2=dict(title='Resp', overlaying='y', side='right'),
    legend=dict(x=0, y=1.1),
    height=400
)
fig.show()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\si

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\si

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\si

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.start()
  File "c:\Users\Kjæreng\anaconda3\Lib\si

AttributeError: _ARRAY_API not found


Processing slp01a
HRV (SDNN): 0.064, RMSSD: 0.030, NN50: 316, pNN50: 4.1%
RSA (RR-Resp korrelasjon): -0.539
Resp (mean ± std): 67.894 ± 19.979

Processing slp01b
HRV (SDNN): 0.087, RMSSD: 0.031, NN50: 692, pNN50: 6.0%
RSA (RR-Resp korrelasjon): -0.230
Resp (mean ± std): 77.715 ± 24.479

Processing slp02a
HRV (SDNN): 0.130, RMSSD: 0.130, NN50: 3038, pNN50: 19.9%
RSA (RR-Resp korrelasjon): 0.065
Resp (mean ± std): 107.454 ± 28.255

Processing slp02b
HRV (SDNN): 0.117, RMSSD: 0.105, NN50: 1729, pNN50: 15.8%
RSA (RR-Resp korrelasjon): -0.022
Resp (mean ± std): 114.271 ± 22.548
